In [1]:
!pip install catboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 1.8 MB/s eta 0:00:56
    --------------------------------------- 1.8/101.7 MB 2.8 MB/s eta 0:00:36
   - -------------------------------------- 2.6/101.7 MB 3.2 MB/s eta 0:00:31
   - -------------------------------------- 4.5/101.7 MB 4.5 MB/s eta 0:00:22
   - -------------------------------------- 5.0/101.7 MB 4.3 MB/s eta 0:00:23
   -- ------------------------------------- 6.8/101.7 MB 4.6 MB/s eta 0:00:21
   --- ------------------------------------ 8.9/101.7 MB 5.4 MB/s eta 0:00:18
   ---- -------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from catboost import CatBoostClassifier

# 1. Load the Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# 2. Split the dataset (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Initialize the CatBoost Classifier
# loss_function='MultiClass' tells the model we are predicting 3 classes (setosa, versicolor, virginica)
# verbose=0 suppresses the long textual printouts of every single tree iteration
catboost_model = CatBoostClassifier(
    iterations=100, 
    learning_rate=0.1, 
    depth=6, 
    loss_function='MultiClass', 
    verbose=0,
    random_seed=42
)

# 4. Train the model
catboost_model.fit(X_train, y_train)

# 5. Make predictions on the test set
y_pred = catboost_model.predict(X_test)

# 6. Print Evaluation Metrics
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


Accuracy Score: 1.0000

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        19
  versicolor       1.00      1.00      1.00        13
   virginica       1.00      1.00      1.00        13

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



In [ ]:
import pandas as pd
df = pd.read_csv("Clean_Health.csv")
dff = pd.read_csv("Clean_Health_2.csv")
dff["New_Gender"] = df["New_Gender"]
dff.to_csv("Clean_Health_2.csv")
dff.columns.to_list()

['Unnamed: 0',
 'Name',
 'Age',
 'Gender',
 'Blood Type',
 'Medical Condition',
 'Date of Admission',
 'Doctor',
 'Hospital',
 'Insurance Provider',
 'Billing Amount',
 'Room Number',
 'Admission Type',
 'Discharge Date',
 'Medication',
 'Test Results',
 'Stay_Dur',
 'New_Gender']

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from catboost import CatBoostClassifier

# 1. Load the dataset (Make sure df matches below)
df = pd.read_csv("Clean_Health_2.csv")

# 2. Separate into features (X) and target (y)
X = df[[
    "Age",
    "Billing Amount",
    "New_Gender",
    "Admission Type",
    "Stay_Dur",          
    "Medical Condition", 
    "Medication"
]]
y = df["Test Results"]

# 3. Explicitly list your categorical string columns
categorical_features = ["Gender","Admission Type", "Medical Condition", "Medication"]

# 4. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 5. Initialize CatBoost
catboost_model = CatBoostClassifier(
    iterations=100, 
    learning_rate=0.1, 
    depth=6, 
    loss_function='MultiClass', 
    verbose=0,
    random_seed=42
)

# 6. Train the model (Crucial: passed the cat_features here!)
catboost_model.fit(X_train, y_train, cat_features=categorical_features)

# 7. Make predictions on the test set
y_pred = catboost_model.predict(X_test)

# 8. Print Evaluation Metrics
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
# Removed iris.target_names to let sklearn automatically discover your medical test result labels
print(classification_report(y_test, y_pred))


ValueError: list.index(x): x not in list

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from catboost import CatBoostClassifier

# 1. Load the dataset
df = pd.read_csv("Clean_Health_3.csv")

# 2. Separate into features (X) and target (y)
X = df[[
    "Age",
    "Billing Amount",
    "Gender",
    "Admission Type",
    "Stay_Dur",          
    # "Medical Condition", 
    "Medication"
]]
y = df["Test Results"]

# 3. FIX: Dynamically find the numerical index positions of your categorical columns
# This completely bypasses any hidden string space/typo bugs!
categorical_features_names = ["Gender", "Admission Type", "Medical Condition", "Medication"]
categorical_indices = [X.columns.get_loc(col) for col in categorical_features_names if col in X.columns]

# 4. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 5. Initialize CatBoost
catboost_model = CatBoostClassifier(
    iterations=200, 
    learning_rate=0.05, 
    depth=4, 
    loss_function='MultiClass', 
    verbose=0,
    random_seed=42
)

# 6. Train the model using the numeric indices
catboost_model.fit(X_train, y_train, cat_features=categorical_indices)

# 7. Make predictions on the test set
y_pred = catboost_model.predict(X_test)

# 8. Print Evaluation Metrics
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))


Accuracy Score: 0.3415

Classification Report:
              precision    recall  f1-score   support

    Abnormal       0.36      0.20      0.25      5643
Inconclusive       0.33      0.15      0.21      5418
      Normal       0.34      0.67      0.45      5589

    accuracy                           0.34     16650
   macro avg       0.34      0.34      0.30     16650
weighted avg       0.34      0.34      0.30     16650

